# 64. 統一APIによるStudy・Ask-and-Tell・Benchmark

未知の目的関数を少ない評価回数で最小化します。Studyはスカラー目的のBOにおける試行履歴と候補生成を管理します。

## このNotebookで確認すること

- 個別FactoryによるBOループとStudyの責務の違い
- optimize、ask、ask_batch、tell、試行履歴の可視化
- Recommendationからの構成選択、履歴からの再開
- Mixed入力と候補制約、ベンチマークでの比較

関連資料：[モデル選択の理論](../../docs/theory/13_model_selection.md)、[統一APIの利用方法](../../docs/capability_workflow.md)、[個別Factory](62_unified_factories.ipynb)、[Recommendation](63_capability_recommendation.ipynb)。

データ → モデル学習 → posterior → 獲得関数 → 候補提案 → 評価 → 履歴更新の順に進めます。

実行環境には `pip install -e ".[examples,dataframe]"` でNotebook用依存関係とDataFrame出力用のpandasをインストールします。

In [ ]:
import torch
import matplotlib.pyplot as plt
from robotorchan.models import create_model
from robotorchan.acquisition import create_acquisition
from robotorchan.optim import create_optimizer, CandidateConstraints, MixedVariableSpace
from robotorchan.study import create_study, TrialState
from robotorchan.workflow import ProblemSpec, recommend

torch.manual_seed(17)
dtype = torch.double
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)

def objective_tensor(X):
    return (X - 0.3).square()

def objective(trial):
    return (trial.params["x"] - 0.3) ** 2

## 1. 個別FactoryでBOループを構築する

最小化する観測値を符号反転してGPに学習させます。qLogEIは最大化する効用を扱います。

In [ ]:
X = torch.tensor([[0.1], [0.6], [0.9]], dtype=dtype)
Y = objective_tensor(X)
solver = create_optimizer("sobol", raw_samples=32, seed=17)
for step in range(2):
    model = create_model("SingleTaskGP", X, -Y)
    model.fit(optimizer_kwargs={"options": {"maxiter": 20}})
    acq = create_acquisition("qLogExpectedImprovement", model=model, best_f=(-Y).max())
    candidate, value = solver(acq, bounds, q=2)
    X = torch.cat([X, candidate])
    Y = torch.cat([Y, objective_tensor(candidate)])
    print(step, candidate.flatten(), Y.min().item())
grid = torch.linspace(0, 1, 100, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
plt.figure(figsize=(7, 3))
plt.plot(grid.squeeze(), -posterior.mean.squeeze(), label="GP mean before last evaluation")
plt.plot(grid.squeeze(), objective_tensor(grid).squeeze(), label="true objective")
plt.scatter(X.squeeze(), Y.squeeze(), label="observations")
plt.legend()
plt.show()

## 2. Studyによる逐次最適化

Studyは観測値の向き、初期探索、モデル再学習、候補生成、履歴を管理します。目的関数は最小化したい値をそのまま返します。

In [ ]:
settings = dict(
    search_space={"x": (0.0, 1.0)}, direction="minimize",
    model="SingleTaskGP", acquisition="qLogExpectedImprovement",
    optimizer="sobol", optimizer_kwargs={"raw_samples": 32, "seed": 17},
    n_initial_trials=3, seed=17,
)
study = create_study(**settings)
study.optimize(objective, n_trials=5)
print(study.best_params, study.best_value)
assert len(study.trials) == 5
assert all(t.state is TrialState.COMPLETE for t in study.trials)
frame = study.trials_dataframe()
display(frame)
plt.figure(figsize=(7, 3))
plt.plot(frame["number"], frame["value"], "o-", label="observed")
plt.plot(frame["number"], frame["value"].cummin(), label="best")
plt.xlabel("Trial")
plt.ylabel("Objective (minimize)")
plt.legend()
plt.show()

## 3. 外部実験とq-batch

askで条件を取得し、外部の実験結果をtellで登録します。ask_batch(2)では2条件をまとめて取得します。

In [ ]:
trial = study.ask()
print(trial.number, trial.params, trial.state)
study.tell(trial.number, objective(trial))
batch = study.ask_batch(2)
for trial in batch:
    print(trial.number, trial.params)
for trial in batch:
    study.tell(trial, objective(trial))
assert len(study.trials) == 8
print(study.best_value)

## 4. RecommendationからStudyを生成する

Optimizerを直接指定する方法と、Search Strategyを指定する方法を比較します。推薦は静的な適合判定であり、性能順位ではありません。

In [ ]:
spec = ProblemSpec(purpose="bayesian_optimization", q=2)
for use_strategy in (False, True):
    (selection,) = recommend(
        spec, model="SingleTaskGP", acquisition="qLogExpectedImprovement",
        optimizer=None if use_strategy else "sobol",
        search_strategy="SobolSearchStrategy" if use_strategy else None,
    )
    key = "num_samples" if use_strategy else "raw_samples"
    recommended_settings = dict(
        search_space={"x": (0.0, 1.0)}, direction="minimize",
        recommendation=selection, optimizer_kwargs={key: 32, "seed": 17},
        n_initial_trials=3, seed=17,
    )
    recommended = create_study(**recommended_settings)
    recommended.optimize(objective, n_trials=3)
    for trial in recommended.ask_batch(2):
        recommended.tell(trial, objective(trial))
    print(selection, recommended.best_value)
    assert len(recommended.trials) == 5

## 5. 試行履歴から再開する

完了・失敗・実行中・待機中のTrialを引き継げます。initial_trialsは履歴の復元であり、永続ストレージや乱数・探索状態の完全再現とは異なります。

In [ ]:
failed = study.ask()
study.tell(failed, state=TrialState.FAIL)
pending = study.ask()
waiting = study.add_trial({"x": 0.5})
resumed = create_study(**settings, initial_trials=study.trials)
assert resumed.trials == study.trials
assert resumed.best_value == study.best_value
resumed.tell(pending.number, objective(pending))
running = resumed.start_trial(waiting.number)
resumed.tell(running, objective(running))
resumed.optimize(objective, n_trials=1)
assert resumed.trials[-1].number == study.trials[-1].number + 1
display(resumed.trials_dataframe())
print(resumed.best_params, resumed.best_value)

## 6. Mixed入力とバッチ間制約

連続変数xと整数変数countを扱います。2候補のcountの合計を6以上とする制約を初期探索にも適用します。

In [ ]:
mixed_bounds = torch.tensor([[0.0, 1.0], [1.0, 4.0]], dtype=dtype)
space = MixedVariableSpace(bounds=mixed_bounds, integer_dims=(1,))
constraints = CandidateConstraints(
    inequality_constraints=((torch.tensor([[0, 1], [1, 1]]),
                             torch.tensor([1.0, 1.0], dtype=dtype), 6.0),),
)
mixed = create_study(
    search_space={"x": (0.0, 1.0), "count": (1.0, 4.0)},
    variable_space=space, constraints=constraints,
    direction="minimize", n_initial_trials=10, seed=11,
)
batch = mixed.ask_batch(2)
assert sum(t.params["count"] for t in batch) >= 6
for trial in batch:
    assert trial.params["count"].is_integer()
    value = (trial.params["x"] - 0.3)**2 + (trial.params["count"] - 3)**2
    mixed.tell(trial, value)
display(mixed.trials_dataframe())

## 7. Benchmarkによる構成比較

同じ問題・seed・初期点数・評価予算を使用し、qLogEIとqUCBを比較します。小さな評価予算の例であり、手法の優劣を結論づけるものではありません。

In [ ]:
from robotorchan.benchmarks.botorch_strategy import make_botorch_gp_strategy
from robotorchan.benchmarks.config import BenchmarkExperimentConfig
from robotorchan.benchmarks.problem import BenchmarkProblem
from robotorchan.benchmarks.registry import BenchmarkProblemRegistry
from robotorchan.benchmarks.runner import run_benchmark

registry = BenchmarkProblemRegistry()
registry.register("quadratic", lambda: BenchmarkProblem(
    name="quadratic", bounds=bounds, objective=objective_tensor,
    directions=("minimize",), variable_types=("continuous",),
    optimal_value=torch.tensor([0.0], dtype=dtype),
))
plt.figure(figsize=(7, 3))
for name in ("qLogExpectedImprovement", "qUpperConfidenceBound"):
    config = BenchmarkExperimentConfig(
        problem="quadratic", strategy=name, seeds=(3, 7),
        initial_points=4, evaluation_budget=4, q=2,
    )
    strategy = make_botorch_gp_strategy(
        model_factory="SingleTaskGP", acquisition=name,
        optimizer="sobol", raw_samples=16, optimizer_kwargs={"seed": 17},
    )
    results = run_benchmark(config, strategy, registry=registry)
    curves = []
    for result in results:
        assert result.X.shape == (8, 1)
        curves.append(result.Y_observed.squeeze(-1).cummin(dim=0).values)
    mean_curve = torch.stack(curves).mean(dim=0)
    plt.plot(torch.arange(1, 9), mean_curve, "o-", label=name)
    print(name, "final best per seed:", [c[-1].item() for c in curves])
plt.xlabel("Total evaluations (including initial points)")
plt.ylabel("Mean best objective")
plt.legend()
plt.show()

## 8. 実務での使い分け

個別Factoryはモデル・獲得関数・候補生成を細かく制御したい場合に適します。Studyはスカラー目的の試行管理に適します。分類・順序回帰の能動学習や多目的BOは、[Notebook 62](62_unified_factories.ipynb)と[Notebook 63](63_capability_recommendation.ipynb)のように個別APIを組み合わせます。Benchmarkは共通の評価条件で構成を比較するために使用します。